# LeetCode 1507 — Reformat Date bằng Python

**Link đề:** https://leetcode.com/problems/reformat-date/
**Độ khó:** Easy · **Chủ đề:** String (xử lý chuỗi)

```python
class Solution:
    def reformatDate(self, date: str) -> str:
        ...
```

### Mục tiêu học
1. Tách chuỗi thành các phần bằng `str.split()` và gán nhiều biến cùng lúc.
2. Bỏ hậu tố `"st"`, `"nd"`, `"rd"`, `"th"` bằng slicing `day[:-2]`, đổi sang số bằng `int()`.
3. Tra tháng bằng `dict` (hoặc `list.index`), và **đệm số 0** bằng `zfill` / f-string `:02d`.
4. Biết 4 cách giải: dict + f-string, list tháng + `index`, `datetime.strptime`, biểu thức chính quy `re`.
5. Trace, test toàn bộ ngày hợp lệ, tránh các lỗi kinh điển (`"1st"` → `"01"`, `"May"` → `"05"`).

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho chuỗi ngày dạng `"Day Month Year"`, trong đó:
- `Day` thuộc tập `{"1st", "2nd", "3rd", "4th", ..., "30th", "31st"}`.
- `Month` thuộc `{"Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"}`.
- `Year` trong khoảng `[1900, 2100]`.

Chuyển sang định dạng `"YYYY-MM-DD"`:
- `YYYY`: năm 4 chữ số,
- `MM`: tháng 2 chữ số,
- `DD`: ngày 2 chữ số.

| Input | Output |
|---|---|
| `"20th Oct 2052"` | `"2052-10-20"` |
| `"6th Jun 1933"` | `"1933-06-06"` |
| `"26th May 1960"` | `"1960-05-26"` |

**Constraints:** ngày cho trước **luôn hợp lệ**, không cần xử lý lỗi.

Đây là định dạng ISO 8601 (`YYYY-MM-DD`) — chuẩn quốc tế, sắp xếp chuỗi theo thứ tự từ điển cũng chính là sắp theo thời gian.

Ba việc phải làm: (1) tách 3 phần; (2) đổi từng phần; (3) ghép lại với dấu `-`.

## 2. Nền tảng Python cần biết

### 2.1. `split()` — tách chuỗi theo khoảng trắng

`s.split()` (không đối số) tách theo mọi khoảng trắng, trả về **list** các từ. Có thể **unpack** ngay vào 3 biến nếu biết chắc có đúng 3 phần.

In [1]:
date = "20th Oct 2052"
parts = date.split()
print(parts)
day, month, year = parts
print("day =", repr(day), "| month =", repr(month), "| year =", repr(year))

['20th', 'Oct', '2052']
day = '20th' | month = 'Oct' | year = '2052'


`repr(x)` in kèm dấu nháy, giúp thấy rõ đây là **chuỗi** `'2052'` chứ không phải số `2052`.

### 2.2. Slicing `[:-2]` — bỏ 2 ký tự cuối

Mọi hậu tố ngày (`st`, `nd`, `rd`, `th`) đều dài **đúng 2 ký tự** (gợi ý của đề). `s[:-2]` lấy từ đầu tới trước 2 ký tự cuối.

In [2]:
for d in ["1st", "2nd", "3rd", "4th", "11th", "22nd", "31st"]:
    print(f"{d:>5} -> d[:-2] = {d[:-2]!r:>5} | d[-2:] = {d[-2:]!r}")

  1st -> d[:-2] =   '1' | d[-2:] = 'st'
  2nd -> d[:-2] =   '2' | d[-2:] = 'nd'
  3rd -> d[:-2] =   '3' | d[-2:] = 'rd'
  4th -> d[:-2] =   '4' | d[-2:] = 'th'
 11th -> d[:-2] =  '11' | d[-2:] = 'th'
 22nd -> d[:-2] =  '22' | d[-2:] = 'nd'
 31st -> d[:-2] =  '31' | d[-2:] = 'st'


`{x!r}` trong f-string tương đương `repr(x)`.

### 2.3. `int()` và đệm số 0: `zfill`, `:02d`

- `int("6")` → `6`; `int("06")` → `6`.
- `"6".zfill(2)` → `"06"`: đệm `0` bên trái cho đủ độ dài 2 (chuỗi đã đủ dài thì giữ nguyên).
- f-string: `f"{6:02d}"` → `"06"` — `0` là ký tự đệm, `2` là độ rộng, `d` là số nguyên.

In [3]:
print(int("6"), int("06"), int("20"))
print("6".zfill(2), "20".zfill(2), "2052".zfill(2))
print(f"{6:02d}", f"{20:02d}", f"{2052:04d}")
print(f"{'6':0>2}", "<- với chuỗi: ký tự đệm '0', căn phải '>', độ rộng 2")

6 6 20
06 20 2052
06 20 2052
06 <- với chuỗi: ký tự đệm '0', căn phải '>', độ rộng 2


### 2.4. `dict` tra tên tháng

Dict ánh xạ **khóa → giá trị**: `months["Oct"]` cho `"10"`. Tra cứu trung bình `O(1)`.

In [4]:
MONTHS = {"Jan": "01", "Feb": "02", "Mar": "03", "Apr": "04", "May": "05", "Jun": "06",
          "Jul": "07", "Aug": "08", "Sep": "09", "Oct": "10", "Nov": "11", "Dec": "12"}
print(MONTHS["Oct"], MONTHS["May"], len(MONTHS))

10 05 12


### 2.5. Ghép chuỗi: `+`, f-string, `"-".join`

In [5]:
y, m, d = "2052", "10", "20"
print(y + "-" + m + "-" + d)
print(f"{y}-{m}-{d}")
print("-".join([y, m, d]))

2052-10-20
2052-10-20
2052-10-20


## 3. Cách 1 — Trực giác: chuỗi `if/elif` cho tháng, nối chuỗi thủ công

Người mới thường viết 12 nhánh `if` cho tháng, và tự thêm `"0"` khi ngày có 1 chữ số. Đúng, nhưng dài và dễ gõ nhầm.

In [6]:
def reformat_date_ifelse(date: str) -> str:
    day, month, year = date.split()
    d = day[:-2]
    if len(d) == 1:
        d = "0" + d
    if month == "Jan": m = "01"
    elif month == "Feb": m = "02"
    elif month == "Mar": m = "03"
    elif month == "Apr": m = "04"
    elif month == "May": m = "05"
    elif month == "Jun": m = "06"
    elif month == "Jul": m = "07"
    elif month == "Aug": m = "08"
    elif month == "Sep": m = "09"
    elif month == "Oct": m = "10"
    elif month == "Nov": m = "11"
    else: m = "12"
    return year + "-" + m + "-" + d

print(reformat_date_ifelse("20th Oct 2052"))
print(reformat_date_ifelse("6th Jun 1933"))
print(reformat_date_ifelse("26th May 1960"))

2052-10-20
1933-06-06
1960-05-26


## 4. Nút thắt — dữ liệu "nằm trong code"

Nhược điểm của cách 1 không phải tốc độ (chỉ 12 nhánh) mà là **khả năng đọc và bảo trì**: bảng tháng bị trộn vào logic. Tách **dữ liệu** (bảng tháng) ra khỏi **logic** (tách – đổi – ghép) bằng `dict` giúp code ngắn, ít lỗi, dễ kiểm tra.

Insight thứ hai: phần ngày luôn là `số + 2 ký tự hậu tố` → `day[:-2]` là đủ, không cần xét `st/nd/rd/th` riêng.

## 5. Cách 2 — `split` + dict + đệm 0 (lời giải chính)

Mổ xẻ:
1. **Tách:** `day, month, year = date.split()`.
2. **Ngày:** `day[:-2]` bỏ hậu tố → `.zfill(2)` đệm 0.
3. **Tháng:** `MONTHS[month]`.
4. **Năm:** giữ nguyên (đã 4 chữ số vì `1900..2100`).
5. **Ghép:** `f"{year}-{mm}-{dd}"`.

In [7]:
MONTHS = {"Jan": "01", "Feb": "02", "Mar": "03", "Apr": "04", "May": "05", "Jun": "06",
          "Jul": "07", "Aug": "08", "Sep": "09", "Oct": "10", "Nov": "11", "Dec": "12"}

def reformat_date_dict(date: str) -> str:
    day, month, year = date.split()
    dd = day[:-2].zfill(2)       # "6th" -> "6" -> "06"
    mm = MONTHS[month]           # "Jun" -> "06"
    return f"{year}-{mm}-{dd}"

print(reformat_date_dict("20th Oct 2052"))
print(reformat_date_dict("6th Jun 1933"))
print(reformat_date_dict("26th May 1960"))

2052-10-20
1933-06-06
1960-05-26


## 6. TRACE từng bước

| Input | `split()` | `day[:-2]` | `zfill(2)` | `MONTHS[month]` | Output |
|---|---|---|---|---|---|
| `"20th Oct 2052"` | `['20th','Oct','2052']` | `'20'` | `'20'` | `'10'` | `"2052-10-20"` |
| `"6th Jun 1933"` | `['6th','Jun','1933']` | `'6'` | `'06'` | `'06'` | `"1933-06-06"` |
| `"1st Jan 1900"` | `['1st','Jan','1900']` | `'1'` | `'01'` | `'01'` | `"1900-01-01"` |

In [8]:
def trace_reformat(date):
    print(f"input       : {date!r}")
    day, month, year = date.split()
    print(f"split()     : day={day!r}, month={month!r}, year={year!r}")
    raw = day[:-2]
    print(f"day[:-2]    : {raw!r}  (bỏ hậu tố {day[-2:]!r})")
    dd = raw.zfill(2)
    print(f"zfill(2)    : {dd!r}")
    mm = MONTHS[month]
    print(f"MONTHS[{month!r}]: {mm!r}")
    out = f"{year}-{mm}-{dd}"
    print(f"output      : {out!r}")
    return out

trace_reformat("6th Jun 1933")
print()
trace_reformat("31st Dec 2100")

input       : '6th Jun 1933'
split()     : day='6th', month='Jun', year='1933'
day[:-2]    : '6'  (bỏ hậu tố 'th')
zfill(2)    : '06'
MONTHS['Jun']: '06'
output      : '1933-06-06'

input       : '31st Dec 2100'
split()     : day='31st', month='Dec', year='2100'
day[:-2]    : '31'  (bỏ hậu tố 'st')
zfill(2)    : '31'
MONTHS['Dec']: '12'
output      : '2100-12-31'


'2100-12-31'

## 7. Cách 3 — List tháng + `index`, định dạng số bằng `:02d`

Thay vì dict, dùng list 12 tên tháng; `list.index(x)` trả về vị trí **đầu tiên** của `x` (0-based) → cộng 1 ra số tháng. `index` dò tuần tự nên `O(12)` — vẫn là hằng số.

In [9]:
MONTH_NAMES = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
print(MONTH_NAMES.index("Jan"), MONTH_NAMES.index("Oct"))

def reformat_date_index(date: str) -> str:
    day, month, year = date.split()
    d = int(day[:-2])
    m = MONTH_NAMES.index(month) + 1
    return f"{year}-{m:02d}-{d:02d}"

print(reformat_date_index("20th Oct 2052"), reformat_date_index("6th Jun 1933"))

0 9
2052-10-20 1933-06-06


Cũng có thể dựng dict tự động từ list bằng **dict comprehension** + `enumerate(..., start=1)` để không gõ tay 12 cặp:

In [10]:
auto_months = {name: f"{i:02d}" for i, name in enumerate(MONTH_NAMES, start=1)}
print(auto_months)
print(auto_months == MONTHS)

{'Jan': '01', 'Feb': '02', 'Mar': '03', 'Apr': '04', 'May': '05', 'Jun': '06', 'Jul': '07', 'Aug': '08', 'Sep': '09', 'Oct': '10', 'Nov': '11', 'Dec': '12'}
True


## 8. Cách 4 — Thư viện chuẩn `datetime`

`datetime.strptime(chuỗi, định_dạng)` **phân tích** (parse) chuỗi thành đối tượng ngày; `.strftime(định_dạng)` **in** ra theo định dạng mới. Mã định dạng:
- `%d` ngày 2 chữ số, `%b` tên tháng viết tắt tiếng Anh (`Jan`…), `%Y` năm 4 chữ số, `%m` tháng 2 chữ số.

`strptime` không hiểu hậu tố `th/st`, nên vẫn phải bỏ 2 ký tự đó trước. Lưu ý: `%b` phụ thuộc **locale** (ngôn ngữ hệ thống); trên LeetCode và đa số máy là tiếng Anh nên chạy được.

In [11]:
from datetime import datetime

def reformat_date_datetime(date: str) -> str:
    day, month, year = date.split()
    dt = datetime.strptime(f"{day[:-2]} {month} {year}", "%d %b %Y")
    return dt.strftime("%Y-%m-%d")

print(datetime.strptime("6 Jun 1933", "%d %b %Y"))
print(reformat_date_datetime("20th Oct 2052"), reformat_date_datetime("6th Jun 1933"))

1933-06-06 00:00:00
2052-10-20 1933-06-06


Ưu điểm phụ: `strptime` **kiểm tra hợp lệ** — `"31 Feb 2000"` sẽ báo lỗi. Đề đã đảm bảo ngày hợp lệ nên không cần, nhưng trong code thực tế đây là lợi thế lớn.

In [12]:
try:
    datetime.strptime("31 Feb 2000", "%d %b %Y")
except ValueError as e:
    print("ValueError:", e)

ValueError: day is out of range for month


## 9. Cách 5 — Biểu thức chính quy (regex) với `re`

Regex mô tả **mẫu** chuỗi. Mẫu `(\d+)\w\w (\w+) (\d+)`:
- `\d+` một hoặc nhiều chữ số → **nhóm 1** là ngày;
- `\w\w` hai ký tự chữ (hậu tố) — không bắt nhóm;
- `(\w+)` tên tháng → **nhóm 2**;
- `(\d+)` năm → **nhóm 3**.

`re.fullmatch` yêu cầu **toàn bộ** chuỗi khớp mẫu; `.groups()` trả tuple các nhóm. Tiền tố `r"..."` (raw string) để `\d` không bị Python hiểu là ký tự thoát.

In [13]:
import re
m = re.fullmatch(r"(\d+)\w\w (\w+) (\d+)", "20th Oct 2052")
print(m.groups())

def reformat_date_regex(date: str) -> str:
    d, mon, y = re.fullmatch(r"(\d+)\w\w (\w+) (\d+)", date).groups()
    return f"{y}-{MONTHS[mon]}-{int(d):02d}"

print(reformat_date_regex("6th Jun 1933"), reformat_date_regex("26th May 1960"))

('20', 'Oct', '2052')
1933-06-06 1960-05-26


## 10. Vì sao cách 2 đúng?

Theo đề, chuỗi luôn có đúng dạng `Day Month Year` với **một** dấu cách giữa các phần:
- `split()` cho đúng 3 phần theo đúng thứ tự.
- `Day` luôn là `1–2 chữ số + 2 chữ cái hậu tố` → `day[:-2]` là phần số (1 hoặc 2 chữ số); `zfill(2)` biến nó thành đúng 2 chữ số mà không đổi giá trị.
- `Month` là một trong 12 khóa của `MONTHS` → tra luôn thành công, cho đúng tháng 2 chữ số.
- `Year` trong `[1900, 2100]` → luôn có đúng 4 chữ số, giữ nguyên.

Ghép bằng `-` theo thứ tự `năm-tháng-ngày` → đúng định dạng `YYYY-MM-DD`.

## 11. Code nộp LeetCode

In [14]:
class Solution:
    def reformatDate(self, date: str) -> str:
        months = {"Jan": "01", "Feb": "02", "Mar": "03", "Apr": "04",
                  "May": "05", "Jun": "06", "Jul": "07", "Aug": "08",
                  "Sep": "09", "Oct": "10", "Nov": "11", "Dec": "12"}
        day, month, year = date.split()
        return f"{year}-{months[month]}-{day[:-2].zfill(2)}"

sol = Solution()
assert sol.reformatDate("20th Oct 2052") == "2052-10-20"
assert sol.reformatDate("6th Jun 1933") == "1933-06-06"
assert sol.reformatDate("26th May 1960") == "1960-05-26"
print("PASS 3 ví dụ đề")

PASS 3 ví dụ đề


## 12. Phân tích Time / Space

Độ dài chuỗi input tối đa khoảng 13 ký tự (`"31st Dec 2100"`) — cố định. Vậy mọi cách đều `O(1)` thời gian và bộ nhớ. Nếu muốn nói theo độ dài chuỗi `L`: `split`, slicing, ghép đều `O(L)`.

| Cách | Thời gian | Bộ nhớ | Ưu / nhược |
|---|---|---|---|
| 1. `if/elif` | `O(1)` | `O(1)` | dài, dễ gõ nhầm |
| 2. dict + `zfill` | `O(1)` | `O(1)` (dict 12 phần tử) | gọn, rõ — **nên dùng** |
| 3. list + `index` | `O(1)` (≤ 12 phép so sánh) | `O(1)` | không cần gõ số tháng |
| 4. `datetime` | `O(1)` | `O(1)` | kiểm tra hợp lệ, phụ thuộc locale |
| 5. regex | `O(1)` | `O(1)` | mạnh khi định dạng phức tạp, khó đọc với người mới |

## 13. Kiểm thử: mẫu đề, biên, kiểm tra toàn bộ miền input

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name:<24} -> {actual}")

tests = [
    ("đề 1",                 "20th Oct 2052", "2052-10-20"),
    ("đề 2 (ngày 1 chữ số)", "6th Jun 1933",  "1933-06-06"),
    ("đề 3",                 "26th May 1960", "1960-05-26"),
    ("nhỏ nhất",             "1st Jan 1900",  "1900-01-01"),
    ("lớn nhất",             "31st Dec 2100", "2100-12-31"),
    ("hậu tố nd",            "2nd Feb 2000",  "2000-02-02"),
    ("hậu tố rd",            "3rd Mar 2001",  "2001-03-03"),
    ("11th (không phải 11st)", "11th Nov 2011", "2011-11-11"),
    ("22nd",                 "22nd Aug 1999", "1999-08-22"),
]
for name, inp, exp in tests:
    for f in (reformat_date_ifelse, reformat_date_dict, reformat_date_index, reformat_date_datetime, reformat_date_regex):
        assert f(inp) == exp, (f.__name__, inp)
    check(name, Solution().reformatDate(inp), exp)

PASS  đề 1                     -> 2052-10-20
PASS  đề 2 (ngày 1 chữ số)     -> 1933-06-06
PASS  đề 3                     -> 1960-05-26
PASS  nhỏ nhất                 -> 1900-01-01
PASS  lớn nhất                 -> 2100-12-31
PASS  hậu tố nd                -> 2000-02-02
PASS  hậu tố rd                -> 2001-03-03
PASS  11th (không phải 11st)   -> 2011-11-11
PASS  22nd                     -> 1999-08-22


In [16]:
# Kiểm tra TOÀN BỘ ngày hợp lệ 1900..2100 so với datetime (chuẩn tham chiếu)
from datetime import date as Date, timedelta

def suffix(d):
    if 11 <= d <= 13:
        return "th"
    return {1: "st", 2: "nd", 3: "rd"}.get(d % 10, "th")

cur, end, count = Date(1900, 1, 1), Date(2100, 12, 31), 0
sol = Solution()
while cur <= end:
    text = f"{cur.day}{suffix(cur.day)} {MONTH_NAMES[cur.month - 1]} {cur.year}"
    assert sol.reformatDate(text) == cur.isoformat(), text
    assert reformat_date_index(text) == cur.isoformat()
    cur += timedelta(days=1)
    count += 1
print(f"Đã kiểm tra {count:,} ngày từ 1900-01-01 tới 2100-12-31. PASS")
print("ví dụ hậu tố:", [f"{d}{suffix(d)}" for d in [1, 2, 3, 4, 11, 12, 13, 21, 22, 23, 31]])

Đã kiểm tra 73,414 ngày từ 1900-01-01 tới 2100-12-31. PASS
ví dụ hậu tố: ['1st', '2nd', '3rd', '4th', '11th', '12th', '13th', '21st', '22nd', '23rd', '31st']


`date.isoformat()` của thư viện chuẩn trả về đúng chuỗi `YYYY-MM-DD` → dùng làm "đáp án chuẩn" để đối chiếu. `timedelta(days=1)` là khoảng thời gian 1 ngày, cộng vào để đi tới ngày kế tiếp.

`dict.get(key, mặc_định)` trả giá trị nếu có khóa, ngược lại trả `mặc_định` — không báo lỗi như `dict[key]`.

## 14. Lỗi người mới hay gặp

### Lỗi 1: Quên đệm 0 cho ngày
```python
return f"{year}-{mm}-{day[:-2]}"   # SAI: "6th Jun 1933" -> "1933-06-6"
```

### Lỗi 2: Bảng tháng không đệm 0 (`"May": "5"`) → `"1960-5-26"`.

### Lỗi 3: Bỏ hậu tố bằng `day[:-1]` hoặc `day[0]`
- `day[:-1]` còn dư một chữ cái: `"20t"`.
- `day[0]` chỉ lấy 1 chữ số: `"20th"` → `"2"`.

### Lỗi 4: `strip("stndrh")` để bỏ hậu tố
`str.strip(chars)` xóa **mọi** ký tự thuộc tập `chars` ở hai đầu — ở đây vẫn chạy đúng vì phần số không chứa chữ cái, nhưng dễ hiểu nhầm `strip("th")` là "xóa chuỗi con `th`". Ví dụ `"thinkth".strip("th")` cho `"ink"` chứ không phải `"think"`.

### Lỗi 5: Đổi năm sang `int` rồi quên định dạng lại — với năm `1900..2100` vẫn ổn, nhưng nếu đề cho năm `< 1000` thì `f"{int(year)}"` mất số 0 đầu. Giữ nguyên chuỗi năm là an toàn nhất.

In [17]:
day, month, year = "6th Jun 1933".split()
print("lỗi 1:", f"{year}-{MONTHS[month]}-{day[:-2]}")
bad_months = {"May": "5"}
print("lỗi 2:", f"1960-{bad_months['May']}-26")
print("lỗi 3a:", "20th"[:-1], "| lỗi 3b:", "20th"[0])
print("lỗi 4 (strip hoạt động theo TẬP ký tự):", "thinkth".strip("th"), "| '22nd'.strip('stndrh') =", "22nd".strip("stndrh"))
print("lỗi 5:", f"{int('0999')}", "vs", f"{int('0999'):04d}")

lỗi 1: 1933-06-6
lỗi 2: 1960-5-26
lỗi 3a: 20t | lỗi 3b: 2
lỗi 4 (strip hoạt động theo TẬP ký tự): ink | '22nd'.strip('stndrh') = 22
lỗi 5: 999 vs 0999


## 15. Mẹo nhận dạng pattern

- Chuỗi có cấu trúc **cố định, ngăn bởi dấu cách** → `split()` rồi unpack.
- Cần **ánh xạ tên → mã** → `dict` (dữ liệu tách khỏi logic).
- Cần số có **độ rộng cố định** → `zfill(k)` hoặc `f"{x:0kd}"`.
- Liên quan ngày giờ trong code thật → nghĩ ngay tới `datetime` (`strptime` / `strftime`).
- Định dạng phức tạp, nhiều biến thể → `re` (regex).

Họ bài tương tự: 1154 Day of the Year, 1185 Day of the Week, 1360 Number of Days Between Two Dates, 2224 Minimum Number of Operations to Convert Time.

### Biến thể

### Biến thể A — Chiều ngược lại: `"YYYY-MM-DD"` → `"Day Month Year"`
Cần sinh hậu tố đúng tiếng Anh: `1st, 2nd, 3rd`, nhưng `11th, 12th, 13th`, và `21st, 22nd, 23rd`.

In [18]:
def to_long_format(iso):
    y, m, d = iso.split("-")
    d = int(d)
    return f"{d}{suffix(d)} {MONTH_NAMES[int(m) - 1]} {y}"

print(to_long_format("2052-10-20"), "|", to_long_format("1933-06-01"), "|", to_long_format("2000-02-12"))

20th Oct 2052 | 1st Jun 1933 | 12th Feb 2000


`split("-")` tách theo dấu `-` thay vì khoảng trắng.

### Biến thể B — Day of the Year (LeetCode 1154)
Từ `"YYYY-MM-DD"` tính đó là ngày thứ mấy trong năm (nhớ năm nhuận). Bạn sẽ làm ở bài tập 6.

## 16. Luyện tập — 6 bài từ dễ đến nâng cao

### Bài 1 (đoán output)
Không chạy, đoán `reformat_date_dict("3rd Sep 2024")`.

In [19]:
def exercise_predict() -> str:
    # TODO
    pass

# assert exercise_predict() == reformat_date_dict("3rd Sep 2024")

### Bài 2 (điền chỗ trống)

In [20]:
def exercise_reformat_date(date: str) -> str:
    months = {"Jan": "01", "Feb": "02", "Mar": "03", "Apr": "04", "May": "05", "Jun": "06",
              "Jul": "07", "Aug": "08", "Sep": "09", "Oct": "10", "Nov": "11", "Dec": "12"}
    # TODO 1: tách day, month, year
    # TODO 2: dd = ?   (bỏ hậu tố, đệm 0)
    # TODO 3: return chuỗi YYYY-MM-DD
    pass

# assert exercise_reformat_date("20th Oct 2052") == "2052-10-20"
# assert exercise_reformat_date("6th Jun 1933") == "1933-06-06"

### Bài 3 (hậu tố tiếng Anh)
Viết `exercise_suffix(d)` cho `1 <= d <= 31` trả về `"st"`, `"nd"`, `"rd"` hoặc `"th"`.

In [21]:
def exercise_suffix(d: int) -> str:
    # TODO: cẩn thận 11, 12, 13
    pass

# assert [exercise_suffix(d) for d in [1, 2, 3, 4, 11, 12, 13, 21, 22, 23, 31]] == \
#        ["st", "nd", "rd", "th", "th", "th", "th", "st", "nd", "rd", "st"]

### Bài 4 (định dạng Mỹ)
Chuyển `"Day Month Year"` sang `"MM/DD/YYYY"`. Ví dụ `"6th Jun 1933"` → `"06/06/1933"`.

In [22]:
def exercise_us_format(date: str) -> str:
    # TODO
    pass

# assert exercise_us_format("6th Jun 1933") == "06/06/1933"
# assert exercise_us_format("20th Oct 2052") == "10/20/2052"

### Bài 5 (sắp xếp ngày)
Cho list các chuỗi `"Day Month Year"`, trả về list đó **sắp theo thời gian** tăng dần (giữ nguyên định dạng gốc). Gợi ý: định dạng ISO sắp xếp theo chuỗi là đúng thứ tự thời gian.

In [23]:
def exercise_sort_dates(dates: list[str]) -> list[str]:
    # TODO: sorted(..., key=?)
    pass

# assert exercise_sort_dates(["6th Jun 1933", "20th Oct 2052", "1st Jan 1933"]) == \
#        ["1st Jan 1933", "6th Jun 1933", "20th Oct 2052"]

### Bài 6 (mini challenge — Day of the Year, LeetCode 1154)
Cho `"YYYY-MM-DD"`, trả về số thứ tự của ngày trong năm. Năm nhuận: chia hết cho 4 nhưng không chia hết cho 100, **hoặc** chia hết cho 400. Không dùng `datetime`.

In [24]:
def exercise_day_of_year(iso: str) -> int:
    # TODO
    pass

# assert exercise_day_of_year("2019-01-09") == 9
# assert exercise_day_of_year("2019-02-10") == 41
# assert exercise_day_of_year("2000-12-31") == 366
# assert exercise_day_of_year("1900-03-01") == 60

### Gợi ý ba mức

**Bài 1** — M1: hậu tố `rd` dài 2. M2: ngày `"3"` → `"03"`. M3: tháng `Sep` là `"09"`.

**Bài 2** — M1: tách – đổi – ghép. M2: `day, month, year = date.split()`. M3: `dd = day[:-2].zfill(2)`, `return f"{year}-{months[month]}-{dd}"`.

**Bài 3** — M1: 11, 12, 13 luôn là `th`. M2: còn lại xét chữ số hàng đơn vị `d % 10`. M3: `{1: "st", 2: "nd", 3: "rd"}.get(d % 10, "th")`.

**Bài 4** — M1: giống bài gốc, chỉ đổi thứ tự ghép. M2: dấu `/`. M3: `f"{mm}/{dd}/{year}"`.

**Bài 5** — M1: biến mỗi ngày thành khóa so sánh được. M2: khóa là chuỗi ISO. M3: `sorted(dates, key=reformat_date_dict)`.

**Bài 6** — M1: cộng số ngày của các tháng **trước** tháng hiện tại, rồi cộng ngày. M2: `days = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]`, năm nhuận thì `days[1] = 29`. M3: `sum(days[:m - 1]) + d`.

## 17. Đáp án tham khảo (tự làm trước rồi mới xem)

In [25]:
def answer_predict():
    return "2024-09-03"

def answer_reformat_date(date):
    months = {"Jan": "01", "Feb": "02", "Mar": "03", "Apr": "04", "May": "05", "Jun": "06",
              "Jul": "07", "Aug": "08", "Sep": "09", "Oct": "10", "Nov": "11", "Dec": "12"}
    day, month, year = date.split()
    dd = day[:-2].zfill(2)
    return f"{year}-{months[month]}-{dd}"

def answer_suffix(d):
    if 11 <= d <= 13:
        return "th"
    return {1: "st", 2: "nd", 3: "rd"}.get(d % 10, "th")

def answer_us_format(date):
    year, mm, dd = answer_reformat_date(date).split("-")
    return f"{mm}/{dd}/{year}"

def answer_sort_dates(dates):
    return sorted(dates, key=answer_reformat_date)

def answer_day_of_year(iso):
    y, m, d = map(int, iso.split("-"))
    days = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    if (y % 4 == 0 and y % 100 != 0) or y % 400 == 0:
        days[1] = 29
    return sum(days[:m - 1]) + d

assert answer_predict() == reformat_date_dict("3rd Sep 2024")
assert answer_reformat_date("20th Oct 2052") == "2052-10-20"
assert answer_reformat_date("6th Jun 1933") == "1933-06-06"
assert [answer_suffix(d) for d in [1, 2, 3, 4, 11, 12, 13, 21, 22, 23, 31]] == \
       ["st", "nd", "rd", "th", "th", "th", "th", "st", "nd", "rd", "st"]
assert answer_us_format("6th Jun 1933") == "06/06/1933"
assert answer_us_format("20th Oct 2052") == "10/20/2052"
assert answer_sort_dates(["6th Jun 1933", "20th Oct 2052", "1st Jan 1933"]) == \
       ["1st Jan 1933", "6th Jun 1933", "20th Oct 2052"]
assert answer_day_of_year("2019-01-09") == 9
assert answer_day_of_year("2019-02-10") == 41
assert answer_day_of_year("2000-12-31") == 366
assert answer_day_of_year("1900-03-01") == 60
# đối chiếu Day of the Year với datetime trên nhiều năm
cur = Date(1899, 1, 1)
while cur <= Date(2101, 12, 31):
    assert answer_day_of_year(cur.isoformat()) == cur.timetuple().tm_yday
    cur += timedelta(days=1)
print("map(int, ['2019','01','09']) ->", list(map(int, ["2019", "01", "09"])))
print("Tất cả đáp án PASS!")

map(int, ['2019','01','09']) -> [2019, 1, 9]
Tất cả đáp án PASS!


`map(int, list_chuỗi)` áp dụng `int` cho từng phần tử — tiện để đổi `"2019-01-09".split("-")` thành 3 số.

## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def reformatDate(self, date: str) -> str:
        months = {"Jan": "01", "Feb": "02", "Mar": "03", "Apr": "04",
                  "May": "05", "Jun": "06", "Jul": "07", "Aug": "08",
                  "Sep": "09", "Oct": "10", "Nov": "11", "Dec": "12"}
        day, month, year = date.split()              # tách
        return f"{year}-{months[month]}-{day[:-2].zfill(2)}"   # đổi + ghép
```

**Checklist trước khi submit:**
- [ ] Bỏ đúng **2** ký tự hậu tố (`[:-2]`)?
- [ ] Ngày được **đệm 0** (`zfill(2)` hoặc `:02d`)?
- [ ] Bảng tháng có đủ 12 tháng và đều 2 chữ số?
- [ ] Thứ tự ghép là **năm-tháng-ngày**, ngăn bởi `-`?
- [ ] Đã test ngày 1 chữ số (`"1st"`, `"6th"`) và tháng ≤ 9?